# 1. 通过transformers认识大模型的结构

In [3]:
from transformers import pipeline

pipe = pipeline(task="text-generation", model=r"F:\03Models\Qwen3-4B", device="cpu", dtype=torch.bfloat16)
message = [
    {"role":"user", "content":"你是谁？"}
]

outputs = pipe(message)

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Device set to use cuda:0


In [4]:
outputs

[{'generated_text': [{'role': 'user', 'content': '你是谁？'},
   {'role': 'assistant',
    'content': '<think>\n好的，用户问我是谁。首先，我需要明确用户的需求。他们可能是在测试我的身份，或者想了解我的功能。作为通义千问，我应该以友好和专业的态度回答。\n\n接下来，我需要介绍自己的身份，说明我是阿里巴巴集团旗下的通义实验室研发的AI助手。要提到我的主要功能，比如回答问题、创作内容等，让用户知道我能提供什么帮助。\n\n然后，要强调我的特点，比如多语言支持、广泛的知识库，以及处理各种任务的能力。这样用户能了解我的优势，可能更愿意使用我。\n\n同时，我应该邀请用户提问或寻求帮助，鼓励他们继续互动。保持语气亲切，避免过于正式，让用户感到轻松。\n\n还要注意不要使用复杂的技术术语，保持回答简洁明了。可能需要检查是否有遗漏的重要信息，比如我的训练数据截止时间，但用户可能不需要这么详细的信息，所以可以简要带过。\n\n最后，确保回答符合用户的需求，既提供足够的信息，又不显得冗长。保持自然流畅，让用户感受到我的友好和专业。\n</think>\n\n我是通义千问，阿里巴巴集团旗下的通义实验室研发的AI助手。我能够回答问题、创作内容、提供帮助，并支持多种'}]}]

In [5]:
type(pipe)

transformers.pipelines.text_generation.TextGenerationPipeline

- transformers的XXXPipeline的工作流：
    - preprocess
    - forward
    - postprocess

In [6]:
msg = [{"role":"user", "content": "你是谁"}]
# 把消息包装成一个Chat对象
from transformers.pipelines.text_generation import Chat   # transformers自己包装的一个类。
chat = Chat(msg)
print(chat)
print(chat.messages)

inputs = pipe.preprocess(chat)
print("*" * 80)
print(inputs)
outputs = pipe.forward(inputs)
print("*" * 80)
print(outputs)
results = pipe.postprocess(outputs)
print("*" * 80)
print(results)

[{'role': 'user', 'content': '你是谁'}]
********************************************************************************
{'input_ids': tensor([[151644,    872,    198, 105043, 100165, 151645,    198, 151644,  77091,
            198]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1]]), 'prompt_text': <transformers.pipelines.text_generation.Chat object at 0x00000158062BF250>}
********************************************************************************
{'generated_sequence': tensor([[[151644,    872,    198, 105043, 100165, 151645,    198, 151644,
           77091,    198, 151667,    198,  99692,   3837,  20002,  56007,
          104198, 100165,   1773, 101140,   3837,  35946,  85106, 100692,
           20002, 104378,   1773,  87267,  99650,  99172,  99794,  97611,
          101294,   3837, 100631,  99172,  81167,  35946,  64471,  20412,
          106168, 105149,   9370,  15469, 104949,   1773, 100345, 108355,
          105051, 100022,   3837,  20002,  87267,  99461,  99392, 10

In [7]:
type(pipe.model)

transformers.models.qwen3.modeling_qwen3.Qwen3ForCausalLM

In [8]:
type(pipe.model.config)

transformers.models.qwen3.configuration_qwen3.Qwen3Config

In [10]:
type(pipe.processor)

NoneType

In [11]:
type(pipe.image_processor)

NoneType

In [12]:
type(pipe.feature_extractor)

NoneType

In [13]:
type(pipe.tokenizer)

transformers.models.qwen2.tokenization_qwen2_fast.Qwen2TokenizerFast

In [15]:
# pipe.tokenizer.get_vocab()

In [1]:
from transformers import Qwen2TokenizerFast, Qwen3ForCausalLM

model = Qwen3ForCausalLM.from_pretrained(r"F:\03Models\Qwen3-4B")
tokenizer = Qwen2TokenizerFast.from_pretrained(r"F:\03Models\Qwen3-4B")

Skipping import of cpp extensions due to incompatible torch version 2.8.0+cu128 for torchao version 0.16.0             Please see https://github.com/pytorch/ao/issues/2919 for more info
W0410 11:34:18.859000 880 site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

In [2]:
# 1. 预处理
msg = [{"role":"user", "content":"35加45等于多少"}]
inputs = tokenizer.apply_chat_template(
    msg,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt"
)
print(inputs)

{'input_ids': tensor([[151644,    872,     18,     20,     19,     20, 151645, 151644,  77091]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1]])}


In [ ]:
# 2. 推理
model.to("cuda")
inputs =inputs.to("cuda")
outputs = model.generate(**inputs, max_new_tokens=512)  # 不是直接调用对象
print(outputs)

In [ ]:
outputs.logits.shape

In [ ]:
# 3. 后处理
print(outputs[0])
results = tokenizer.decode(outputs[0][inputs["input_ids"].shape[-1]:])
# print(results)

In [ ]:
print(results)

-----

- 训练：
    - 数据集。